# Orchestration Patterns

**Prerequisites:** `03_communication_state_and_collaboration.ipynb`

Multi-agent systems get taught as a list of architectures — supervisor, peer-to-peer, blackboard, group chat, pipeline, debate, hierarchical, hybrid — as though each were a different kind of thing. They are not. They are answers to **three different questions**, and every system answers all three:

| | The question | Answers |
|---|---|---|
| **Organisation** | who can talk to whom | a supervisor over a roster; peers holding each other; everyone on one board |
| **Control** | who runs next | a manager that picks; an event, when something is written; a fixed order |
| **Execution shape** | what shape the work moves in | one after another; all at once, then combined; turn and turn about |

Read that way, the list collapses. "Group chat with a manager" is a board plus a manager picking speakers — an organisation and a control, not a third architecture. "Debate" is turn-taking with the turns fixed in advance. "Hybrid" is not a type at all: it is what every real system is, because the three questions are answered separately.

The one thing worth saying twice: **shared state never decides who runs next.** A board says where information lives. Something else has to say who acts on it, and that something is the control question.

**By the end you'll have:**
- Placed everything this chapter already built on the two axes
- Run the same case peer-to-peer, with no supervisor in the middle
- Run a group chat where a manager picks each speaker, and the same board driven by an **event** instead, with nobody choosing
- Run a debate, where each side has to answer the other, and seen what that adds over asking both at once
- A table of which combinations are worth using, and why "hybrid" is a property rather than a pattern

## Setup

## Where this chapter already is

Nothing below is new machinery. It is the same four specialists, rearranged — and what has been built so far already covers half the table:

| Built in | Organisation | Control | Execution shape |
|---|---|---|---|
| **nb2** supervisor over a roster | hierarchical | the supervisor's model picks | one at a time, as it decides |
| **nb2** handoff (`Chat`) | peer-to-peer, after the transfer | whoever holds the conversation | sequential |
| **nb3** board (`Shared`) | blackboard | **none** — the cells ran in the order they were written | pipeline, by accident |
| **nb3** three opinions | none: they never see each other | all at once, decided by the code | fan-out, then aggregation |

The third row is the one worth staring at. `03`'s board section looked like a system, and it had no control mechanism at all: engineering ran because the cell above support's cell said so. That is a pipeline — a fixed order, written by hand — and it is a perfectly good answer to the control question. It is just easy to mistake the board for the thing that made it work.

## Organisation: peer-to-peer

The supervisor in notebook 2 sat between everyone. Peer-to-peer removes it: the agent that has the question holds the agent that can answer it, as a tool.

That is all `coordination.peers` does — `messages.as_form_tool` for each colleague, added to the agent's own registry:

```python
def peers(agent, others, log=None):
    return replace(agent, tools=Registry([*agent.tools, *(as_form_tool(o, log) for o in others)]))
```

Here it is on the case notebook 2 got wrong. Support gets the customer's report directly, with no supervisor anywhere, and engineering as a colleague it can call.

Support held `ask_engineer` and `ask_lawyer`, looked the serial up with its own tool, and **called neither colleague**. Two calls in total. The reply is decent — it leads with "stop charging right now" and offers an inspection, which is better than notebook 2's "for now, you're safe" — but it still tells the customer their serial "isn't in the batch we're recalling" without anyone checking the one thing that mattered, the cell lot.

**Peer-to-peer removes the supervisor. It does not create the conversation.** Taking the middle agent out changes who *can* talk to whom, and nothing about who *does*.

### The same pair, with the rule written down

Notebook 2 measured this on the smaller model: a rule in an agent's own instructions changed its behaviour 8 times out of 8, and the same rule in a tool description 0 times out of 8. Peer-to-peer does not escape it. Support holds `ask_engineer` either way — what decides whether it uses it is whether the job it was given says to.

### How often does it ask, without being told?

One run of each is an anecdote. The plain task is cheap (two calls), so here it is four times over, counting only whether support reached for `ask_engineer` at all.

With the rule in the task, support asked engineering first, and the request it wrote carried the serial, the symptom and the batch — a self-contained question, of the kind notebook 2 had to learn to write. Engineering's form came back `confident: yes, visibility: internal`, saying the unit uses lot VC‑7731 and could have the fault with low probability. Support's reply then told the customer exactly that: the fault can happen outside the recalled batch, so start a free return.

It cost 5 calls against 2, so roughly 2.5× — the price of asking.

**And without the rule it asked nobody, 0 times out of 4.**

That is worth pausing on, because this notebook runs on the *bigger* model. Notebook 2 found on gpt-oss-20b that a routing rule works in the supervisor's prompt and not in a tool description, and left open whether a larger model would read descriptions better. On this evidence the shape of the problem survives the size change: gpt-oss-120b also does not reach for a colleague it was not told to reach for. The two measurements are not the same experiment — notebook 2 compared *where* the rule was written, this compares *having* the rule against not having it — but they point the same way. **What an agent may do lives in its tools; what it should do has to be in the job you give it.**

## Control: a manager picks, or an event fires

Now the same four agents on a board, with the two live ways of deciding who acts.

**A manager.** `coordination.GroupChat` gives one agent the roster and the transcript and asks it, through a form, who should speak next — and lets it say `done`. Everyone speaks into the same transcript, and reads the same board.

The manager picked engineering, and engineering did the work rather than offering an opinion: it called `cell_lot`, recorded `cell_lot: VC-7731` and `risk: yes` on the board, and gave both the customer guidance and the wording for the regulator. Then the manager said `done` — after **one** speaker.

That is a group chat working correctly, and it is worth noticing how little it looks like a group. A manager is a control mechanism, not a promise that everyone participates: asked to stop as soon as the question has an answer somebody can act on, it stopped when the first specialist answered it.

Two smaller things:

- **The manager's decision is a form, not prose.** `Pick(speaker, why)` means the program routes on a field instead of parsing a sentence, and a speaker nobody has — including the literal `done` — stops the chat rather than being invented into existence.
- **Deciding is cheap; speaking is not.** Each `Pick` is one call. The 8 calls in this cell are mostly engineering's tool use.

An earlier run of this cell, on the smaller model, went differently in a way worth recording: three agents were picked in turn and all three answered from the brief without touching a tool, so the meeting concluded that nothing needed to change and the board stayed empty. `GroupChat` now tells each speaker to check what it is unsure of and to record what others need — which is a prompt fixing a prompt problem, not a pattern fixing it.

**An event.** The other way needs no manager and no model call to decide anything. `Shared.watch` runs a callback whenever a field is written, so the lawyer can be wired to the moment engineering records a risk:

```python
board.watch("risk", when_risk_recorded)
```

Nobody chooses. Engineering does its job, and the act of recording what it found is what starts the lawyer.

Nothing chose. Engineering recorded its two fields; the act of writing `risk` fired the watcher; the watcher ran the lawyer with the board in its context; the lawyer recorded `regulator_update`. The board ends with all four fields filled, and **no model call was spent on deciding who acts** — the wiring decided, in Python, before the run started.

Against the manager above: 7 calls versus 8, and near-identical tokens, but the group chat spent some of its calls choosing while this spent all of them working. What the manager buys for that difference is the ability to react to something the wiring did not anticipate. What the event buys is that it always happens.

**One thing I got wrong here is worth keeping.** The first version of this cell also decided *whether* to fire, by looking for "7731" or "recall" in the recorded text. That run recorded `risk: no`, the condition did not match, and the lawyer never ran — a safety-relevant event lost to a substring test against prose. Code that reacts to a field should either fire on the write and let an agent judge (what it does now), or read a field the code can actually compare: a level, a flag, an enum. A board whose fields are sentences can be read by agents and only guessed at by code.

Chains are the other thing to watch: a watcher can write, and that write can fire another watcher. Easy to build, easy to loop, which is notebook 5's problem.

**A fixed order.** The third control is the one with no runtime decision in it at all: `coordination.pipeline` runs a list of agents in the order given, each seeing what the last produced. Notebook 3's board section was exactly this — engineering, then support, in the order the cells appeared — which is why it needs no demonstration here beyond naming it. A pipeline is the cheapest scheduler there is, and the least able to react to what it finds.

## Execution shape: debate

Notebook 3 asked three specialists the same question at the same time, and they never saw each other's answers. Engineering said yes, the lawyer and PR said no from the brief alone, and the aggregation had to sort it out afterwards.

A debate changes the shape rather than the roster: the same two disagreeing agents, in turns, each answering what the other just said.

Both sides argued to widen the recall. The engineer from the cell lot; the lawyer from the same lot plus the duty it creates. Round two elaborated rather than contested.

So the debate produced **agreement**, and that is a real result rather than a failed demo: given the same facts, on this model, both specialists read them the same way. Compare notebook 3, where the same question asked in parallel on the smaller model split three ways, with the lawyer and PR answering from the brief alone. **A debate makes each side answer the other. It cannot manufacture a disagreement that the evidence does not support** — and when there is nothing to fight about, you have paid four agent turns to find that out.

It is the most expensive cell in the notebook: 16,085 tokens and 74 seconds, because every turn carries the whole argument so far.

The judge is the part to look at twice. It answered `yes`, `confident: True`, and `evidence: none`. It is right, and it checked nothing — it read an argument. `aggregate.decisive(judge)` would therefore refuse to let it decide, which is the rule working as intended: a judge summarising other people's evidence is not the same as an agent that looked. If you want a debate to end in a decision you can defend, the verdict needs to cite which side's evidence it rests on.

Also worth noting for later: the engineer's argument cites the litigation threat from the brief's internal section. Inside the company that is fine. A debate transcript is an internal document, and anything that quotes one has to be treated that way.

## What it cost

- **Deciding is cheap, doing is not.** A manager's `Pick` is one call; a speaker's turn is several. The event-driven version spends nothing at all on deciding.
- **Asking a colleague costs about what you would expect:** 2 calls became 5 once support actually asked engineering.
- **The debate is the outlier**, at 16,085 tokens for four turns, because each turn re-reads the whole argument. Two agents in parallel (notebook 3) answered the same question for a third of that, and learned less about each other's reasons.
- **Trials are affordable when the case is small:** four plain peer runs cost 11,152 tokens and turned an anecdote into 0 of 4.
- **The whole notebook is about 59,000 tokens on gpt-oss-120b.** The same cells were first run on gpt-oss-20b for roughly 35,000, and the outputs were unusable for this purpose: agents held tools without using them, so every pattern looked the same. That is the honest reason for the model switch, and the reason notebooks 1–3 stay where their findings were measured.

## Hybrids, and why they are not a fourth category

Everything in this chapter, on the two axes:

| | Organisation | Control | Execution shape |
|---|---|---|---|
| nb2 supervisor | hierarchical | the supervisor's model picks | one at a time |
| nb2 handoff | peer-to-peer after the transfer | whoever holds the conversation | sequential |
| nb3 board + two cells | blackboard | fixed order (the cells) | pipeline |
| nb3 three opinions | isolated | code, all at once | fan-out → aggregation |
| nb4 peers | peer-to-peer | the agent holding the question | as needed |
| nb4 group chat | blackboard + transcript | a manager picks | turn by turn |
| nb4 event | blackboard | an event on a write | triggered |
| nb4 debate | two agents, in turn | fixed alternation | debate → judge |

Read down the columns and the combinations write themselves, which is all "hybrid" ever meant:

- **hierarchical + parallel** — a supervisor that fans work out and aggregates it (notebook 2's supervisor with notebook 3's `opinions`).
- **hierarchical + peer-to-peer** — a supervisor over teams whose members ask each other directly instead of going up and back down.
- **blackboard + manager, with events underneath** — a manager for the judgement calls, a watcher for the things that must always happen, like the regulator being told.
- **pipeline of debates** — a fixed sequence of stages, one of which is two agents arguing.

None of those is a new architecture. Each is one choice per column.

## What this notebook found

- **Shared state never decides who runs next.** Notebook 3's board looked like a system and its control was the order of the cells. Name the control mechanism, or you have one by accident.
- **Peer-to-peer removes the middle agent, not the need to ask.** Support held its colleagues as tools and asked nobody, 0 times out of 4, on the larger model. The instruction to ask belongs in the job, not in the tool list.
- **A manager is a scheduler that can react**, and it stops early when the first specialist answers the question. Its decision should be a form so the program can act on it.
- **An event is a scheduler that cannot react and never forgets.** Fire on the write and let an agent judge; a condition written against prose misses `risk: no`.
- **A debate makes each side answer the other**, at several times the cost of asking them in parallel, and it cannot create disagreement that the evidence does not support. Its judge reads an argument rather than checking anything, and a rule like `decisive` will rightly refuse it.

Notebook 5 builds one system out of all of this, and then breaks it: a worker that fails, a worker that loops, a reply that cannot be used. Every pattern here has a way of going wrong that the happy path hides.